# 5 mmol Static XAS: Non-Finite VLS Check

Small notebook to check where VLS `NaN`/`inf` values come from in the processed static-XAS H5 file.

Important detail: `/vls` is stored as a rectangular array `(energy, n_shots_max, pixel)`, but each energy only has `n_shots[ie]` valid rows. Rows after `n_shots[ie]` are intentional `NaN` padding.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

mpl_cache = Path(tempfile.gettempdir()) / "glycine26-matplotlib-cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160})


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "analysis" / "scripts" / "static_xas_nonfinite_check.py").exists():
            return parent
    raise RuntimeError("Could not find repo root")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPTS_DIR = REPO_ROOT / "analysis" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import config as path_config
import static_xas_nonfinite_check as nfc

REPO_ROOT

## Parameters

Change `RUN_NO` if you want to audit another processed static-XAS file.

In [ ]:
RUN_NO = 58794
INPUT_H5 = Path(path_config.COMBINED_DIR).parent / "xas_static" / f"run{RUN_NO}_static_xas.h5"

XLIM = (271.0, 286.0)
CHUNK_SIZE = 20_000
PADDING_PROBE_ROWS = 16

INPUT_H5

## Scan

The scan only treats rows `0:n_shots[ie]` as recorded spectra. Padding rows are counted separately.

In [ ]:
scan = nfc.scan_static_xas_nonfinite(
    INPUT_H5,
    chunk_size=CHUNK_SIZE,
    padding_probe_rows=PADDING_PROBE_ROWS,
)

nfc.print_scan_summary(scan)

## Per-Energy Counts

`VLS bad shots` means recorded spectra with at least one non-finite VLS pixel. `GMD bad shots` means recorded rows where GMD is `NaN` or `inf`.

In [ ]:
print("energy_eV  recorded  padded  VLS_bad_shots  GMD_bad_shots  VLS_finite_frac  GMD_finite_frac")
for ie, energy in enumerate(scan.nominal_energies):
    print(
        f"{energy:8.2f}  "
        f"{scan.n_shots[ie]:8d}  "
        f"{scan.padded_rows[ie]:6d}  "
        f"{scan.valid_vls_bad_shots[ie]:13d}  "
        f"{scan.valid_gmd_bad_shots[ie]:13d}  "
        f"{scan.valid_vls_finite_fraction[ie]:15.6f}  "
        f"{scan.valid_gmd_finite_fraction[ie]:15.6f}"
    )

## Recorded Rows vs Padding

This plot separates actual recorded rows from rectangular-array padding. If VLS finite fraction is 1 but padding rows are large, whole-cube scans will report many VLS `NaN`s for a storage-format reason.

In [ ]:
nfc.plot_storage_and_finite_counts(scan, xlim=XLIM)
plt.show()

## Valid VLS Pixel Map

This heatmap ignores padding and only checks VLS pixels inside `0:n_shots[ie]`.

In [ ]:
nfc.plot_valid_vls_nonfinite_heatmap(scan)
plt.show()

## Padding Probe

This checks a small number of rows immediately after `n_shots[ie]`. Values near 1 mean those rows are intentionally non-finite padding.

In [ ]:
nfc.plot_padding_probe(scan, xlim=XLIM)
plt.show()

## Example Recorded Spectra With Bad VLS Pixels

If the valid-region scan finds real VLS `NaN`/`inf` pixels, they will appear here. Otherwise this cell prints a simple no-example message.

In [ ]:
nfc.plot_first_nonfinite_vls_examples(scan)
plt.show()

## Working Interpretation

In this processed layout, VLS `NaN`s usually mean one of two different things:

1. Rows after `n_shots[ie]`: intentional padding used to store all energies in one rectangular H5 dataset.
2. Rows inside `n_shots[ie]`: real recorded spectra with invalid detector or processing values.

For run 58794, the quick audit should show no non-finite VLS pixels inside valid rows. With the mapped hall GMD source (`/ml/gmd_hall_channels[..., 0]`), the valid rows should also have finite GMD. If root `/gmd` is inspected directly in an older enriched file, it may show artificial NaNs from the historical bunch-index mapping bug.